In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_CreateRunContext
# MAGIC Creates one run_id for this workflow execution and publishes it
# MAGIC as a task value for every downstream task.

# COMMAND ----------

import uuid
from datetime import datetime, timezone

dbutils.widgets.text("connection_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_prefix", "full")

connection_id  = dbutils.widgets.get("connection_id").strip()
supplied_run_id = dbutils.widgets.get("run_id").strip()
run_prefix     = dbutils.widgets.get("run_prefix").strip() or "full"

if not connection_id:
    raise ValueError("connection_id is required")

if supplied_run_id:
    run_id = supplied_run_id
    print("Using supplied run_id")
else:
    stamp  = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{run_prefix}_{stamp}_{uuid.uuid4().hex[:8]}"
    print("Generated new run_id")

print(f"connection_id : {connection_id}")
print(f"run_id        : {run_id}")

dbutils.jobs.taskValues.set(key="run_id", value=run_id)
dbutils.jobs.taskValues.set(key="connection_id", value=connection_id)

dbutils.notebook.exit(run_id)